In [ ]:
# imports and paths

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.dates as mdates
from matplotlib.colors import TwoSlopeNorm
from windrose import WindroseAxes

PROCESSED_DIR = Path("processed_data")
SCADA_PATH = PROCESSED_DIR / "penmanshiel_scada_complete.parquet"
STATIC_PATH = PROCESSED_DIR / "penmanshiel_static.parquet"

OUTPUT_DIR = Path("figures/eda")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_TURBINES = 14
TURBINE_RATED_POWER_KW = 2050
FARM_RATED_POWER_KW = N_TURBINES * TURBINE_RATED_POWER_KW
EQUAL_TURBINE_SHARE_PCT = 100 / N_TURBINES

LABEL_FONTSIZE = 14
TICK_FONTSIZE = 12
LEGEND_FONTSIZE = 10


In [ ]:
# load the cleaned complete-timestamp dataset

scada = pd.read_parquet(SCADA_PATH).copy()
static = pd.read_parquet(STATIC_PATH).copy()

scada["timestamp"] = pd.to_datetime(
    scada["timestamp"],
    utc=True,
)

scada["turbine_id"] = scada["turbine_id"].astype(int)
static["turbine_id"] = static["turbine_id"].astype(int)

turbine_ids = sorted(scada["turbine_id"].unique())

print(f"turbines: {turbine_ids}")
print(f"farm timestamps: {scada['timestamp'].nunique():,}")


In [ ]:
# helper functions

def turbine_label(turbine_id):
    return f"WT{int(turbine_id):02d}"


def circular_mean_degrees(direction_wide):
    direction_radians = np.deg2rad(direction_wide)

    mean_sine = np.nanmean(
        np.sin(direction_radians),
        axis=1,
    )

    mean_cosine = np.nanmean(
        np.cos(direction_radians),
        axis=1,
    )

    mean_direction = np.rad2deg(
        np.arctan2(mean_sine, mean_cosine)
    )

    return np.mod(mean_direction, 360)


def angular_distance_degrees(angle, centre):
    return (angle - centre + 180) % 360 - 180


def format_axes(ax, grid_axis="both"):
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.0)
        spine.set_color("black")

    ax.tick_params(
        axis="both",
        which="both",
        direction="out",
        length=5,
        width=1,
        labelsize=TICK_FONTSIZE,
        top=False,
        right=False,
    )

    ax.grid(
        axis=grid_axis,
        alpha=0.20,
        linewidth=0.8,
    )

    ax.set_axisbelow(True)


In [ ]:
# create farm-level and turbine-wide tables used throughout the eda

power_wide = (
    scada
    .pivot(
        index="timestamp",
        columns="turbine_id",
        values="power_kw",
    )
    .reindex(columns=turbine_ids)
    .sort_index()
)

wind_speed_wide = (
    scada
    .pivot(
        index="timestamp",
        columns="turbine_id",
        values="wind_speed",
    )
    .reindex(columns=turbine_ids)
    .sort_index()
)

wind_direction_wide = (
    scada
    .pivot(
        index="timestamp",
        columns="turbine_id",
        values="wind_dir",
    )
    .reindex(columns=turbine_ids)
    .sort_index()
)

farm_data = pd.DataFrame(
    index=power_wide.index
)

farm_data["farm_power_kw"] = power_wide.sum(axis=1)

farm_data["reference_wind_speed"] = (
    wind_speed_wide.max(axis=1)
)

farm_data["wind_direction"] = circular_mean_degrees(
    wind_direction_wide
)

print(farm_data.describe().round(2))


In [ ]:
# figure 3.1 turbine layout

fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(
    static["x_m"],
    static["y_m"],
    s=70,
    color="steelblue",
    marker="x",
    linewidths=2,
    zorder=3,
)

for _, turbine in static.iterrows():
    ax.annotate(
        turbine_label(turbine["turbine_id"]),
        xy=(turbine["x_m"], turbine["y_m"]),
        xytext=(6, 6),
        textcoords="offset points",
        fontsize=10,
    )

ax.set_xlabel(
    "East–west position (m)",
    fontsize=LABEL_FONTSIZE,
)

ax.set_ylabel(
    "North–south position (m)",
    fontsize=LABEL_FONTSIZE,
)

ax.set_aspect(
    "equal",
    adjustable="box",
)

ax.margins(
    x=0.12,
    y=0.12,
)

format_axes(ax)

fig.tight_layout()

fig.savefig(
    OUTPUT_DIR / "fig_3_1_turbine_layout.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


In [ ]:
# figure 3.2 wind rose

fig = plt.figure(figsize=(7, 7))
ax = WindroseAxes.from_ax(fig=fig)

ax.bar(
    scada["wind_dir"],
    scada["wind_speed"],
    normed=True,
    opening=0.85,
    edgecolor="white",
    linewidth=0.5,
)

ax.set_thetagrids(
    angles=[0, 45, 90, 135, 180, 225, 270, 315],
    labels=["N", "NE", "E", "SE", "S", "SW", "W", "NW"],
    fontsize=TICK_FONTSIZE,
)

ax.tick_params(
    axis="y",
    labelsize=10,
)

ax.set_rlabel_position(67.5)

ax.grid(
    alpha=0.25,
    linewidth=0.8,
)

legend = ax.set_legend(
    title="Wind speed (m/s)",
    loc="lower left",
    bbox_to_anchor=(-0.10, -0.03),
    fontsize=9,
)

legend.get_title().set_fontsize(10)

fig.tight_layout()

fig.savefig(
    OUTPUT_DIR / "fig_3_2_wind_rose.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


In [ ]:
# figure 3.3 nacelle wind-speed distribution

wind_speed = scada["wind_speed"].dropna()

wind_speed_limit = np.ceil(
    wind_speed.quantile(0.995)
)

wind_speed_bins = np.arange(
    0,
    wind_speed_limit + 0.5,
    0.5,
)

mean_wind_speed = wind_speed.mean()
median_wind_speed = wind_speed.median()

fig, ax = plt.subplots(figsize=(7.5, 5))

ax.hist(
    wind_speed,
    bins=wind_speed_bins,
    color="steelblue",
    alpha=0.85,
    edgecolor="white",
    linewidth=0.6,
)

ax.axvline(
    mean_wind_speed,
    color="black",
    linestyle="--",
    linewidth=1.5,
    label=f"Mean = {mean_wind_speed:.2f} m/s",
)

ax.axvline(
    median_wind_speed,
    color="black",
    linestyle=":",
    linewidth=1.5,
    label=f"Median = {median_wind_speed:.2f} m/s",
)

ax.set_xlabel(
    "Wind speed (m/s)",
    fontsize=LABEL_FONTSIZE,
)

ax.set_ylabel(
    "Frequency",
    fontsize=LABEL_FONTSIZE,
)

ax.set_xlim(
    0,
    wind_speed_limit,
)

format_axes(
    ax,
    grid_axis="y",
)

ax.legend(
    frameon=False,
    fontsize=LEGEND_FONTSIZE,
)

fig.tight_layout()

fig.savefig(
    OUTPUT_DIR / "fig_3_3_wind_speed_distribution.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print(f"mean wind speed: {mean_wind_speed:.2f} m/s")
print(f"median wind speed: {median_wind_speed:.2f} m/s")


In [ ]:
# figure 3.4 farm power as a function of the farm reference wind speed

fig, ax = plt.subplots(figsize=(9, 5))

ax.scatter(
    farm_data["reference_wind_speed"],
    farm_data["farm_power_kw"] / 1000,
    s=2,
    alpha=0.10,
    color="steelblue",
)

ax.axhline(
    FARM_RATED_POWER_KW / 1000,
    color="black",
    linestyle="--",
    linewidth=1.2,
    label=f"Rated capacity ({FARM_RATED_POWER_KW / 1000:.1f} MW)",
)

ax.set_xlabel(
    "Farm reference wind speed (m/s)",
    fontsize=LABEL_FONTSIZE,
)

ax.set_ylabel(
    "Total farm power (MW)",
    fontsize=LABEL_FONTSIZE,
)

format_axes(ax)

ax.legend(
    frameon=False,
    fontsize=LEGEND_FONTSIZE,
)

fig.tight_layout()

fig.savefig(
    OUTPUT_DIR / "fig_3_4_farm_power_curve.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


In [ ]:
# figure 3.5 farm power variation with wind direction at 7 to 9 m/s

direction_subset = farm_data[
    farm_data["reference_wind_speed"].between(
        7.0,
        9.0,
        inclusive="left",
    )
].copy()

direction_bin_edges = np.arange(
    0,
    365,
    5,
)

direction_subset["direction_bin"] = pd.cut(
    direction_subset["wind_direction"],
    bins=direction_bin_edges,
    include_lowest=True,
)

direction_summary = (
    direction_subset
    .groupby(
        "direction_bin",
        observed=True,
    )["farm_power_kw"]
    .agg(
        median="median",
        p25=lambda values: values.quantile(0.25),
        p75=lambda values: values.quantile(0.75),
        count="size",
    )
)

direction_summary["direction_centre"] = [
    interval.mid
    for interval in direction_summary.index
]

direction_summary = direction_summary[
    direction_summary["count"] >= 50
].copy()

fig, ax = plt.subplots(figsize=(9, 5))

ax.fill_between(
    direction_summary["direction_centre"],
    direction_summary["p25"] / 1000,
    direction_summary["p75"] / 1000,
    color="steelblue",
    alpha=0.20,
    label="25th–75th percentile",
)

ax.plot(
    direction_summary["direction_centre"],
    direction_summary["median"] / 1000,
    color="steelblue",
    linewidth=2,
    label="Median farm power",
)

ax.set_xlim(0, 360)
ax.set_xticks(np.arange(0, 361, 45))

ax.set_xlabel(
    "Wind direction (°)",
    fontsize=LABEL_FONTSIZE,
)

ax.set_ylabel(
    "Total farm power (MW)",
    fontsize=LABEL_FONTSIZE,
)

format_axes(ax)

ax.legend(
    frameon=False,
    fontsize=LEGEND_FONTSIZE,
)

fig.tight_layout()

fig.savefig(
    OUTPUT_DIR / "fig_3_5_farm_power_by_direction.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


In [ ]:
# figure 3.6 mean turbine contribution to farm power

positive_farm_power = (
    farm_data["farm_power_kw"] > 0
)

power_share_pct = (
    power_wide.loc[positive_farm_power]
    .div(
        farm_data.loc[
            positive_farm_power,
            "farm_power_kw",
        ],
        axis=0,
    )
    * 100
)

share_summary = pd.DataFrame(
    {
        "mean_share_pct": power_share_pct.mean(),
        "median_share_pct": power_share_pct.median(),
        "p05_share_pct": power_share_pct.quantile(0.05),
        "p95_share_pct": power_share_pct.quantile(0.95),
    }
)

share_summary["p90_range_pct"] = (
    share_summary["p95_share_pct"]
    - share_summary["p05_share_pct"]
)

share_summary = share_summary.sort_values(
    "mean_share_pct",
    ascending=True,
)

lower_error = (
    share_summary["mean_share_pct"]
    - share_summary["p05_share_pct"]
)

upper_error = (
    share_summary["p95_share_pct"]
    - share_summary["mean_share_pct"]
)

x_error = np.vstack(
    [lower_error.values, upper_error.values]
)

fig, ax = plt.subplots(figsize=(8, 5))

ax.barh(
    [
        turbine_label(turbine_id)
        for turbine_id in share_summary.index
    ],
    share_summary["mean_share_pct"],
    xerr=x_error,
    capsize=3,
    color="steelblue",
    alpha=0.85,
)

ax.axvline(
    EQUAL_TURBINE_SHARE_PCT,
    color="black",
    linestyle="--",
    linewidth=1.2,
    label=f"Equal share ({EQUAL_TURBINE_SHARE_PCT:.2f}%)",
)

ax.set_xlabel(
    "Share of total farm power (%)",
    fontsize=LABEL_FONTSIZE,
)

ax.set_ylabel(
    "Turbine",
    fontsize=LABEL_FONTSIZE,
)

format_axes(
    ax,
    grid_axis="x",
)

ax.legend(
    loc="lower right",
    frameon=False,
    fontsize=LEGEND_FONTSIZE,
)

fig.tight_layout()

fig.savefig(
    OUTPUT_DIR / "fig_3_6_turbine_power_contribution.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print(
    share_summary
    .sort_values("mean_share_pct", ascending=False)
    .round(2)
    .to_string()
)


In [ ]:
# figure 3.7 turbine contributions over the illustrative two-day period

area_start = pd.Timestamp(
    "2019-04-01",
    tz="UTC",
)

area_end = pd.Timestamp(
    "2019-04-03",
    tz="UTC",
)

area_share = power_share_pct.loc[
    area_start:area_end
].copy()

area_share_hourly = (
    area_share
    .resample("1h")
    .mean()
    .dropna(how="any")
)

colours = cm.tab20(
    np.linspace(
        0,
        1,
        len(turbine_ids),
    )
)

fig, ax = plt.subplots(figsize=(9, 5))

ax.stackplot(
    area_share_hourly.index,
    *[
        area_share_hourly[turbine_id].values
        for turbine_id in turbine_ids
    ],
    labels=[
        turbine_label(turbine_id)
        for turbine_id in turbine_ids
    ],
    colors=colours,
)

ax.set_ylim(0, 100)

ax.set_xlabel(
    "Time",
    fontsize=LABEL_FONTSIZE,
)

ax.set_ylabel(
    "Share of total farm power (%)",
    fontsize=LABEL_FONTSIZE,
)

ax.xaxis.set_major_formatter(
    mdates.DateFormatter("%d %b")
)

format_axes(
    ax,
    grid_axis="y",
)

ax.legend(
    ncol=2,
    fontsize=8,
    frameon=False,
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
)

fig.tight_layout()

fig.savefig(
    OUTPUT_DIR / "fig_3_7_turbine_share_area.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

largest_turbine = area_share_hourly.idxmax(axis=1)

number_largest = largest_turbine.nunique()

number_changes = (
    largest_turbine
    .ne(largest_turbine.shift())
    .sum()
    - 1
)

area_ranges = pd.DataFrame(
    {
        "min_share_pct": area_share_hourly.min(),
        "max_share_pct": area_share_hourly.max(),
    }
)

print(f"hourly observations: {len(area_share_hourly)}")
print(f"turbines largest at least once: {number_largest}")
print(f"changes in largest turbine: {number_changes}")
print(area_ranges.round(2).to_string())


In [ ]:
# figure 3.8 turbine power-curve behaviour

curve_data = scada[
    scada["wind_speed"].between(
        3.0,
        14.0,
        inclusive="both",
    )
].copy()

curve_data["wind_speed_bin"] = (
    np.round(
        curve_data["wind_speed"] / 0.5
    )
    * 0.5
)

median_power_curves = (
    curve_data
    .groupby(
        ["wind_speed_bin", "turbine_id"]
    )["power_kw"]
    .median()
    .unstack("turbine_id")
    .sort_index()
)

median_across_turbines = (
    median_power_curves.median(axis=1)
)

relative_power_pct = (
    median_power_curves
    .sub(
        median_across_turbines,
        axis=0,
    )
    .div(
        median_across_turbines,
        axis=0,
    )
    * 100
)

relative_power_pct = relative_power_pct[
    median_across_turbines >= 100
]

curve_colours = cm.tab20(
    np.linspace(
        0,
        1,
        len(turbine_ids),
    )
)

fig, axes = plt.subplots(
    2,
    1,
    figsize=(8.5, 8),
    sharex=True,
)

for colour, turbine_id in zip(
    curve_colours,
    turbine_ids,
):
    axes[0].plot(
        median_power_curves.index,
        median_power_curves[turbine_id],
        color=colour,
        linewidth=1.2,
        alpha=0.85,
        label=turbine_label(turbine_id),
    )

    axes[1].plot(
        relative_power_pct.index,
        relative_power_pct[turbine_id],
        color=colour,
        linewidth=1.2,
        alpha=0.85,
    )

axes[1].axhline(
    0,
    color="black",
    linestyle="--",
    linewidth=1.2,
)

axes[0].set_ylabel(
    "Turbine power (kW)",
    fontsize=LABEL_FONTSIZE,
)

axes[1].set_xlabel(
    "Turbine wind speed (m/s)",
    fontsize=LABEL_FONTSIZE,
)

axes[1].set_ylabel(
    "Deviation from median power (%)",
    fontsize=LABEL_FONTSIZE,
)

axes[0].set_xlim(3, 14)
axes[0].set_ylim(0, 2200)

for ax in axes:
    format_axes(ax)

axes[0].legend(
    fontsize=8,
    ncol=2,
    frameon=False,
)

fig.tight_layout()

fig.savefig(
    OUTPUT_DIR / "fig_3_8_turbine_power_curves.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

gap_rows = []

for wind_speed in [5.0, 8.0, 12.0]:
    turbine_powers = median_power_curves.loc[wind_speed].dropna()

    highest_power = turbine_powers.max()
    lowest_power = turbine_powers.min()

    gap_rows.append(
        {
            "wind_speed_ms": wind_speed,
            "highest_turbine": turbine_label(turbine_powers.idxmax()),
            "lowest_turbine": turbine_label(turbine_powers.idxmin()),
            "gap_pct": (
                (highest_power - lowest_power)
                / lowest_power
                * 100
            ),
        }
    )

power_curve_gap_summary = pd.DataFrame(gap_rows)

print(power_curve_gap_summary.round(2).to_string(index=False))


In [ ]:
# figure 3.9 directional variation in relative turbine wind speed

farm_median_wind_speed = wind_speed_wide.median(axis=1)

relative_wind_speed_pct = (
    wind_speed_wide
    .div(
        farm_median_wind_speed,
        axis=0,
    )
    * 100
)

direction_centres = [0, 180, 225]
direction_half_width = 5

direction_results = {}

for direction_centre in direction_centres:
    direction_distance = angular_distance_degrees(
        farm_data["wind_direction"],
        direction_centre,
    )

    direction_mask = (
        np.abs(direction_distance)
        <= direction_half_width
    )

    direction_results[direction_centre] = (
        relative_wind_speed_pct.loc[direction_mask]
        .median(axis=0)
    )

all_direction_values = np.concatenate(
    [
        values.dropna().values
        for values in direction_results.values()
    ]
)

maximum_deviation = max(
    100 - np.nanmin(all_direction_values),
    np.nanmax(all_direction_values) - 100,
)

colour_norm = TwoSlopeNorm(
    vmin=100 - maximum_deviation,
    vcenter=100,
    vmax=100 + maximum_deviation,
)

layout = (
    static
    .set_index("turbine_id")
    .loc[turbine_ids]
)

for direction_centre in direction_centres:
    direction_values = direction_results[
        direction_centre
    ]

    fig, ax = plt.subplots(figsize=(8, 6))

    scatter = ax.scatter(
        layout["x_m"],
        layout["y_m"],
        c=direction_values.loc[turbine_ids],
        cmap="coolwarm",
        norm=colour_norm,
        s=280,
        edgecolor="black",
        linewidth=0.8,
        zorder=3,
    )

    for turbine_id in turbine_ids:
        ax.annotate(
            (
                f"{turbine_label(turbine_id)}\n"
                f"{direction_values.loc[turbine_id]:.1f}%"
            ),
            xy=(
                layout.loc[turbine_id, "x_m"],
                layout.loc[turbine_id, "y_m"],
            ),
            xytext=(7, 7),
            textcoords="offset points",
            fontsize=9,
        )

    ax.set_xlabel(
        "East–west position (m)",
        fontsize=LABEL_FONTSIZE,
    )

    ax.set_ylabel(
        "North–south position (m)",
        fontsize=LABEL_FONTSIZE,
    )

    ax.set_aspect(
        "equal",
        adjustable="box",
    )

    format_axes(ax)

    colourbar = fig.colorbar(
        scatter,
        ax=ax,
        pad=0.02,
    )

    colourbar.set_label(
        "Wind speed relative to farm median (%)",
        fontsize=12,
    )

    fig.tight_layout()

    fig.savefig(
        OUTPUT_DIR
        / f"fig_3_9_relative_wind_speed_{direction_centre}deg.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

    print(
        f"\nwind from {direction_centre}° ± {direction_half_width}°"
    )

    print(
        direction_values
        .sort_values()
        .round(2)
        .rename(index=turbine_label)
        .to_string()
    )


In [ ]:
# save the eda summary tables used in the report text

summary_dir = OUTPUT_DIR / "summary_tables"
summary_dir.mkdir(exist_ok=True)

direction_summary.to_csv(
    summary_dir / "farm_power_by_direction.csv"
)

share_summary.to_csv(
    summary_dir / "turbine_power_contributions.csv"
)

area_ranges.to_csv(
    summary_dir / "illustrative_area_share_ranges.csv"
)

power_curve_gap_summary.to_csv(
    summary_dir / "turbine_power_curve_gaps.csv",
    index=False,
)

print(f"saved eda outputs to {OUTPUT_DIR}")
